In [ ]:


import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 1561312


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [ ]:
from pytorch_lightning import seed_everything

seed_everything(SEED, workers=True)

In [ ]:

import torch
import torch.nn.functional as nnf

from tqdm import trange

from PIL import Image




import torch.nn as nn
from torch.utils.data import Dataset


from transformers import AutoTokenizer
from peft.optimizers import create_loraplus_optimizer



In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [ ]:
device = "cuda"

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:

device = "cuda"

In [ ]:
import pandas as pd
import glob

class imageTextDataset(Dataset):
    def __init__(self, folder_path):

        self.pt_files = glob.glob(os.path.join(folder_path, "*.pt"))
       
    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        data = torch.load(self.pt_files[idx], map_location="cpu")

        return data



In [ ]:
# dataset = "/run/media/victor/pessoal/mestrado/dataset/Pathcap/path_cap_clean.csv"

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
device="cuda"

In [ ]:
model_id = "HuggingFaceTB/SmolLM2-135M"

llm_tokenizer = AutoTokenizer.from_pretrained(model_id, device_map=device)



In [ ]:
llm_tokenizer(llm_tokenizer.eos_token)

In [ ]:
llm_tokenizer.pad_token = llm_tokenizer.eos_token 

In [ ]:
llm_tokenizer("teste testando", return_tensors="pt",  padding="max_length", max_length=5)

In [ ]:


# image_dataset =  imageTextDataset(dataset, siglip, siglip_processor, llm_tokenizer)

In [ ]:
# from torch.utils.data import random_split

# train_data, test_data = random_split(image_dataset, [8/10, 2/10], generator=generator)

In [ ]:
train_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/v6_pathcap_processed/train"
test_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/v6_pathcap_processed/test"


In [ ]:
train_data = imageTextDataset(train_dataset) 
test_data = imageTextDataset(test_dataset)

In [ ]:
len(train_data), len(test_data)

In [ ]:
def collate_fn(batch):

  return {
      'image_embeds': torch.stack([x['image_embeds'] for x in batch]),
      'input_ids': torch.stack([x['input_ids'] for x in batch]),
      'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
      'labels': torch.stack([x['labels'] for x in batch]),
}

In [ ]:
from torch.utils.data import DataLoader

# batch_size=18
batch_size=16
num_workers = 1

train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=collate_fn,
        shuffle=True,
        generator=generator,
        persistent_workers=True
)

test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=num_workers,
        generator=generator,
        persistent_workers=True
)

In [ ]:

from transformers import AutoModelForCausalLM

llm = AutoModelForCausalLM.from_pretrained(model_id,  device_map=device)



In [ ]:
llm.train()

In [ ]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
class MLP(nn.Module):
    def pixel_shuffle(self, x):
        bsz, seq, embed_dim = x.size()
        height = width = int(seq**0.5)
        x = x.view(bsz, height, width, embed_dim)
        x = x.view(bsz, height, int(width / self.scale_factor), embed_dim * self.scale_factor)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(bsz, int(width / self.scale_factor), int(height / self.scale_factor), embed_dim * (self.scale_factor**2))
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(bsz, int(seq / (self.scale_factor**2)), embed_dim * (self.scale_factor**2))
        return x

    def forward(self, input):
        input_shuffle = self.pixel_shuffle(input)
        return self.model(input_shuffle)

    def __init__(self, input_size, output_size, scale_factor=2):
        super(MLP, self).__init__()

        self.scale_factor = scale_factor

        self.model = nn.Sequential(
            nn.Linear(input_size* (self.scale_factor**2), output_size), 
   
        )

In [ ]:
import pytorch_lightning as L
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

class PathCaptionModel(L.LightningModule):
    def __init__(self, mlp_shape, llm, warmup_steps = 5933):
        super().__init__()

        self.llm = llm.train().to(self.device)
        self.projection = MLP(input_size=mlp_shape[0], output_size=mlp_shape[1])
        self.projection.train().to(self.device)
        self.warmup_steps = 0
        #self.total_steps = 3000
   

    def forward(self, input):
        image_embeds = input["image_embeds"].to(self.device)  
        input_ids = input["input_ids"].to(self.device)
        attention_mask = input["attention_mask"].to(self.device)
        labels = input["labels"].to(self.device)
        
        image_proj = self.projection(image_embeds)

        token_embeds = self.llm.get_input_embeddings()(input_ids)
        combined_embeds = torch.cat((image_proj, token_embeds), dim=1)

        image_mask = torch.ones(image_proj.shape[:2], dtype=torch.long, device=self.device)
        combined_mask = torch.cat((image_mask, attention_mask), dim=1)

        # Create combined labels (ignore loss for image prefix token)
        prefix_labels = torch.full(image_mask.shape, -100, dtype=torch.long, device=self.device)
        combined_labels = torch.cat((prefix_labels, labels), dim=1)


        labels = input["labels"].to(self.device)

        llm_output = self.llm.forward(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask,
            labels=combined_labels,
            return_dict=True
        )

        return llm_output
        

    def training_step(self, batch, batch_idx):
        outputs = self(batch)
        loss = outputs.loss
        self.log("train_loss", loss)

        return loss
    
    def validation_step(self, batch, batch_idx):
        outputs = self(batch)
        loss = outputs.loss
        self.log("val_loss", loss)

        return loss

    
    def configure_optimizers(self):
        projection_params = list(self.projection.parameters())
        llm_params = [p for p in self.llm.parameters() if p.requires_grad]

        # Add warmup scheduler
        optimizer = torch.optim.AdamW([
            {"params": projection_params, "lr": 1e-3},  
            {"params": llm_params, "lr": 5e-4}  
        ])

        # 1. Warmup scheduler
        def lr_lambda(current_step):
            if current_step < self.warmup_steps:
                return float(current_step) / float(max(1, self.warmup_steps))
            return 1.0

        warmup_scheduler = LambdaLR(optimizer, lr_lambda)


        # main_scheduler = CosineAnnealingLR(
        #     optimizer, 
        #     T_max=self.total_steps - self.warmup_steps,
        #     eta_min=5e-6
        # )

        # lr_scheduler = SequentialLR(
        #     optimizer,
        #     schedulers=[warmup_scheduler, main_scheduler],
        #     milestones=[self.warmup_steps]
        # )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": warmup_scheduler,
                "interval": "step", # Update the scheduler every step
                "frequency": 1,
            },
        }


In [ ]:
mlp_shape = (768,576)

In [ ]:
mlp_shape

In [ ]:
model = PathCaptionModel(mlp_shape, llm)

In [ ]:
model = torch.compile(model)

In [ ]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="/run/media/victor/pessoal/mestrado/codigo/train/logs")
version = logger.version

In [ ]:
filename_template = f"llm-1-phase-epoch{{epoch}}-val_loss{{val_loss:.6f}}-batch{batch_size}-v{version}-seed{SEED}"
    

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

early_stopping_callback = EarlyStopping('val_loss', patience=3)
    
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='/run/media/victor/pessoal/mestrado/codigo/train/checkpoint',
    filename=filename_template,
    save_top_k=1,
    mode="min",
    auto_insert_metric_name=False,
    save_weights_only=True,

)

In [ ]:
from pytorch_lightning.callbacks import LearningRateMonitor

lr_monitor = LearningRateMonitor(logging_interval='epoch')

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

trainer = L.Trainer(max_epochs=-1,
        max_steps=-1,
        accelerator="auto",
        devices="auto",
        log_every_n_steps=1,
        logger=logger,
         precision="bf16-mixed",
         deterministic=True,
         callbacks=[early_stopping_callback, checkpoint_callback, lr_monitor],
         #accumulate_grad_batches=4,
    )

In [ ]:
trainer.fit(model, train_loader, test_loader)